# **Bloques del DFE sin solución simbólica a tiempo: `dfe_timeout`**

Para calcular $\mathcal{R}_0$ hay que evaluar las Jacobianas $F$ y $V$ en el equilibrio libre de infección (DFE). `pyR0compute` resuelve el DFE **por bloques**: separa las ecuaciones de los compartimentos no infectados en componentes fuertemente conexas y las resuelve en orden topológico (ver el cuaderno `03_sistemas_no_lineales.ipynb`).

Algunos bloques acoplados no tienen una solución simbólica razonable, aunque sí la tengan numéricamente. En ese caso `sympy.solve` puede quedarse calculando durante horas, y **no es posible interrumpirlo desde el mismo proceso de Python**.

Desde la versión 0.2.1, cada bloque difícil se resuelve en un **proceso aparte con un límite de tiempo**, `dfe_timeout` (20 s por defecto):

* Si SymPy encuentra la solución a tiempo, el bloque se escribe en forma cerrada, como antes.
* Si no, el proceso se detiene y el bloque queda **implícito**: sus valores son símbolos $X^*$ definidos por $\dot X = 0$. $\mathcal{R}_0$ se entrega en forma compacta en función de esos $X^*$, y `dfe_numeric`, `R0_numeric`, `dfe_stability`, `prcc` y `sobol_indices` los calculan numéricamente.
* Los bloques fáciles (una ecuación de grado $\le 2$) se resuelven directamente, sin costo adicional.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yvan-BM-Git/pyR0compute/blob/main/examples/04_limite_de_tiempo_dfe.ipynb)

In [ ]:
# Instala pyR0compute desde PyPI con el extra [global] (SciPy >= 1.11).
# La versión en desarrollo: %pip install "pyR0compute[global] @ git+https://github.com/Yvan-BM-Git/pyR0compute"
%pip install "pyR0compute[global]>=0.2.1"

In [ ]:
import time                                   # Medir tiempos de cálculo
import warnings                               # Mostrar los avisos de la librería
import numpy as np                            # Cálculo numérico
import sympy as sp                            # Cálculo simbólico
from pyr0compute import R0Model               # Clase principal de la librería
from IPython.display import Markdown, display # Para mostrar ecuaciones en LaTeX

## 1. Modelo intrahuésped con un lazo de regulación inmune

Células blanco $T$, células infectadas $I$ y virus $V$, con tres mediadores de la respuesta innata que forman un **lazo**: el interferón $F$ activa a las células NK $N$ (función de Hill de orden 2), las células NK activan a los macrófagos $M$ y los macrófagos producen interferón. Las células NK eliminan células infectadas y los macrófagos eliminan virus.

$$
\begin{cases}
\dfrac{dT}{dt} = s - d_T T - \beta T V \\[6pt]
\dfrac{dI}{dt} = \beta T V - (\delta + k_N N)\, I \\[6pt]
\dfrac{dV}{dt} = p I - (c + k_M M)\, V \\[6pt]
\dfrac{dN}{dt} = a_N - d_N N + h_N \dfrac{F^2}{F^2 + K_F^2} \\[8pt]
\dfrac{dM}{dt} = a_M - d_M M + h_M \dfrac{N}{N + K_N} \\[8pt]
\dfrac{dF}{dt} = a_F - d_F F + h_F \dfrac{M}{M + K_M}
\end{cases}
$$

Con $I = V = 0$, el DFE tiene dos bloques: $[T]$, con $T^* = s/d_T$, y el lazo $[N, M, F]$. Al eliminar $M$ y $F$, este último se reduce a una cúbica en $N$ con 121 términos en 12 parámetros. `sympy.solve` sobre el sistema acoplado no termina: en la versión 0.2.0 este modelo seguía calculando después de varios minutos.

In [ ]:
lazo = '''
    dT/dt = s - d_T*T - beta*T*V
    dI/dt = beta*T*V - (delta + k_N*N)*I
    dV/dt = p*I - (c + k_M*M)*V
    dN/dt = a_N - d_N*N + h_N*F^2/(F^2 + K_F^2)
    dM/dt = a_M - d_M*M + h_M*N/(N + K_N)
    dF/dt = a_F - d_F*F + h_F*M/(M + K_M)
'''

t0 = time.time()
with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    modelo = R0Model(lazo, infected=["I", "V"], dfe_timeout=10)   # límite de 10 s por bloque
print(f"Modelo construido en {time.time() - t0:.1f} s\n")
for aviso in avisos:
    print("Aviso:", aviso.message)

### Cómo se resolvió cada bloque

`dfe_blocks` informa, en el orden de resolución, el estado de cada bloque: `"closed form"` (forma cerrada), `"time limit"` (no se resolvió dentro del límite) o `"no closed form"` (SymPy no encontró forma cerrada, como en la raíz de una quíntica), y el tiempo dedicado a resolverlo simbólicamente.

In [ ]:
for bloque in modelo.dfe_blocks:
    print(f"{str(bloque['variables']):<18} {bloque['status']:<13} {bloque['seconds']:6.2f} s")

print("\nDFE compacto:", modelo.dfe_compact)
print("Definiciones:", modelo.dfe_definitions)   # None: sin forma cerrada, se calcula numéricamente
print("¿DFE implícito?", modelo.dfe_is_implicit)

### $\mathcal{R}_0$ compacto

Aunque $N^*$ y $M^*$ no tienen forma cerrada, $\mathcal{R}_0$ sí la tiene en función de ellos:

$$
\mathcal{R}_0 = \frac{\beta\, p\, s}{d_T\,(\delta + k_N N^*)\,(c + k_M M^*)}.
$$

El número de células blanco en ausencia de infección es $T^* = s/d_T$. Cada célula infectada produce virus a tasa $p$ durante un tiempo medio $1/(\delta + k_N N^*)$, y cada virión infecta a tasa $\beta T^*$ durante $1/(c + k_M M^*)$. La respuesta innata basal reduce $\mathcal{R}_0$ a través de $N^*$ y $M^*$.

In [ ]:
display(Markdown(f"$$\\mathcal{{R}}_0 = {modelo.latex()}$$"))

El informe paso a paso indica qué valores no se encontraron dentro del límite:

In [ ]:
print(modelo.report())

### Valores numéricos

Valores ilustrativos (no provienen de un ajuste a datos). Para cada conjunto de parámetros, el bloque implícito se resuelve integrando el subsistema no infectado hacia su estado estacionario y refinando con Newton. La función del bloque se compila una sola vez, así que cada evaluación toma milisegundos.

In [ ]:
valores = dict(s=10, d_T=0.1, beta=0.005, delta=0.5, k_N=0.2, p=20, c=3, k_M=0.3,
               a_N=1, d_N=0.5, h_N=2, K_F=1, a_M=1, d_M=0.4, h_M=1.5, K_N=1,
               a_F=0.5, d_F=1, h_F=2, K_M=1)

print("DFE numérico:")
for x, valor in modelo.dfe_numeric(valores).items():
    print(f"  {x} = {valor:.6f}")
print(f"R0 = {modelo.R0_numeric(valores):.6f}")

t0 = time.time()
for _ in range(100):
    modelo.R0_numeric(valores)
print(f"100 evaluaciones de R0 en {time.time() - t0:.2f} s")

estab = modelo.dfe_stability(valores)
print("Condición (A5): valores propios del bloque no infectado =",
      ", ".join(f"{z.real:.4f}" for z in estab["eigenvalues"]),
      "->", "estable" if estab["stable"] else "inestable")

### Umbral

Con (A5), el DFE del sistema completo es estable si $\mathcal{R}_0 < 1$ e inestable si $\mathcal{R}_0 > 1$. Lo verificamos con la mayor parte real de los valores propios del Jacobiano completo ($6\times6$) en el DFE, variando la tasa de infección $\beta$.

In [ ]:
f = sp.Matrix([modelo.equations[v] for v in modelo.variables])
J = f.jacobian(list(modelo.variables))

for beta in [0.002, 0.004, 0.005, 0.01]:
    v = dict(valores, beta=beta)
    Jn = J.subs(modelo.dfe_numeric(v)).subs({s: v[s.name] for s in J.free_symbols - set(modelo.variables)})
    abscisa = np.max(np.linalg.eigvals(np.array(Jn.evalf(), dtype=float)).real)
    print(f"beta = {beta:<6} R0 = {modelo.R0_numeric(v):7.4f}   máx Re(λ) = {abscisa:+.4f}")

### Efecto de la respuesta innata

Como el DFE depende de los parámetros del lazo, $\mathcal{R}_0$ también. Por ejemplo, al aumentar la activación de las células NK por interferón ($h_N$) crecen $N^*$ y, a través del lazo, $M^*$, y $\mathcal{R}_0$ disminuye.

In [ ]:
print(f"{'h_N':>5} {'N*':>9} {'M*':>9} {'R0':>9}")
for h_N in [0, 1, 2, 4, 8]:
    v = dict(valores, h_N=h_N)
    dfe = {x.name: val for x, val in modelo.dfe_numeric(v).items()}
    print(f"{h_N:>5} {dfe['N']:9.4f} {dfe['M']:9.4f} {modelo.R0_numeric(v):9.4f}")

Con $h_N = 0$ el lazo no actúa sobre las células NK y $N^* = a_N/d_N = 2$. Con estos valores, desde $h_N = 4$ la respuesta innata basal basta para que $\mathcal{R}_0 < 1$: la infección no puede establecerse.

### Sensibilidad global

La sensibilidad global funciona igual que con un DFE en forma cerrada; aquí $\mathcal{R}_0$ se calcula numéricamente en cada muestra. Como no hay una expresión simbólica completa de $\mathcal{R}_0$, la monotonía que exige el PRCC no se verifica con derivadas (`monotonic` queda en `None`).

In [ ]:
t0 = time.time()
res = modelo.prcc(baseline=valores, spread=0.2, n=1000, seed=1)   # todos uniformes en ±20 %
print(f"LHS-PRCC (n = 1000) en {time.time() - t0:.1f} s; los 8 parámetros más influyentes:")
tabla = res.as_dict()
for nombre in res.ranking()[:8]:
    print(f"  {nombre:<6} PRCC = {tabla[nombre]['PRCC']:+.3f}")

## 2. Cómo elegir `dfe_timeout`

* **Por defecto (20 s).** Los bloques que SymPy resuelve en forma cerrada tardan desde milisegundos hasta algunos segundos. Por ejemplo, el bloque acoplado $\{H, R\}$ del modelo de 7 ecuaciones del cuaderno 03 toma unos 6 s (ver abajo) y queda en forma cerrada. En un computador más lento un bloque así podría superar el límite; en ese caso no se pierde nada, solo se calcula numéricamente, y basta aumentar `dfe_timeout` para recuperar la forma cerrada.
* **Un valor menor** (por ejemplo 5 s) acelera la construcción de modelos que ya sabes que no tienen solución simbólica útil.
* **Un valor mayor** da más tiempo a SymPy si esperas una forma cerrada.
* **`dfe_timeout=None`** quita el límite y resuelve en el mismo proceso, como en la versión 0.2.0. Úsalo solo si sabes que el bloque termina.

Los modelos con DFE sencillo no cambian. En el SEIR, por ejemplo, todos los bloques se resuelven en forma cerrada en milisegundos:

In [ ]:
seir = R0Model('''
    dS/dt = Lambda - beta*S*I - mu*S
    dE/dt = beta*S*I - (sigma + mu)*E
    dI/dt = sigma*E - (gamma + mu)*I
    dR/dt = gamma*I - mu*R
''', infected=["E", "I"])
for bloque in seir.dfe_blocks:
    print(f"{str(bloque['variables']):<10} {bloque['status']:<12} {bloque['seconds']:.3f} s")
print("R0 =", seir.R0)

El modelo de 7 ecuaciones del cuaderno 03 tampoco cambia: su bloque acoplado $\{H, R\}$ se resuelve en forma cerrada dentro del límite.

In [ ]:
t0 = time.time()
inmune = R0Model('''
    dE/dt = g_E - (d_E + tau_E*V/(V + z_VE))*E
    dI/dt = tau_E*V*E/(V + z_VE) - (d_I + tau_I*C/(C + z_CI))*I
    dV/dt = nu*I - (d_V + tau_V*B/(B + z_BV))*V
    dH/dt = g_H - (d_H - tau_H*I/(I + z_IH) + rho_H*R/(R + z_RH))*H
    dC/dt = g_C - (d_C - tau_C*H/(H + z_HC) + rho_C*R/(R + z_RC))*C
    dB/dt = g_B - (d_B - tau_B*H/(H + z_HB) + rho_B*R/(R + z_RB))*B
    dR/dt = g_R - (d_R - tau_R*H/(H + z_HR) + rho_R)*R
''', infected=["I", "V"])
print(f"Modelo construido en {time.time() - t0:.1f} s")
for bloque in inmune.dfe_blocks:
    print(f"{str(bloque['variables']):<12} {bloque['status']:<12} {bloque['seconds']:.2f} s")